### Imports

In [3]:
import os
import json
import tempfile
import subprocess
from itertools import islice

import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download


### Creating nexar dataframe

In [7]:
# Now that we know the CSV structure, we can transform it according to the user’s requested format.
import json

src_path = "nexar_train.csv"
out_path = "nexar_train_prepared.h5"

df = pd.read_csv(src_path)

# Build the target dataframe
final_df = pd.DataFrame()
final_df["filepath"] = df["filepath"]
final_df["source"] = "huggingface"
final_df["event_time"] = df["time_of_event"]

# Infer collision vs no collision based on presence of time_of_event
final_df["event_description"] = df["time_of_event"].apply(
    lambda x: "collision" if pd.notna(x) else "no collision"
)

final_df["split"] = df["split"] if "split" in df.columns else "train"

# Bundle all other columns into `other_info`
used_cols = {"filepath", "time_of_event", "split"}
other_info_cols = [c for c in df.columns if c not in used_cols]

final_df["other_info"] = df[other_info_cols].to_dict(orient="records")
final_df["other_info"] = final_df["other_info"].apply(json.dumps)

# Save to .h5
final_df.to_hdf(out_path, key="data", mode="w")

In [9]:
out_path = "nexar_train_prepared.h5"

df = pd.read_hdf(out_path, key="data")

df


,filepath,source,event_time,event_description,split,other_info
0,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.500,collision,train,"{""time_of_alert"": 18.633, ""light_conditions"": ..."
1,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.800,collision,train,"{""time_of_alert"": 19.233, ""light_conditions"": ..."
2,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,20.101,collision,train,"{""time_of_alert"": 20.068, ""light_conditions"": ..."
3,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.267,collision,train,"{""time_of_alert"": 18.333, ""light_conditions"": ..."
4,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,21.146,collision,train,"{""time_of_alert"": 18.196, ""light_conditions"": ..."
...,...,...,...,...,...,...
1495,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1496,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1497,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1498,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."


### Template Code

In [ ]:
import os
import cv2
from huggingface_hub import HfFileSystem

# === CONFIG ===
TEMP_DIR = "temp_videos"
os.makedirs(TEMP_DIR, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")  # set via environment variable
fs = HfFileSystem(token=HF_TOKEN)


def load_video(row):
    """
    Given a DataFrame row (with 'source' and 'filepath'),
    downloads the video if source == 'huggingface', saves it locally,
    and returns the local file path.
    """
    source = row["source"]
    filepath = row["filepath"]

    if source == "huggingface":
        # Extract clean filename from the path
        filename = os.path.basename(filepath)
        local_path = os.path.join(TEMP_DIR, filename)

        # Download if not already present
        if not os.path.exists(local_path):
            print(f"Downloading {filepath} from Hugging Face...")
            with fs.open(filepath, "rb") as src, open(local_path, "wb") as dst:
                dst.write(src.read())

        return os.path.abspath(local_path)

    else:
        raise NotImplementedError(f"Source '{source}' not supported yet.")


def script1(video_path):
    """
    Simple video player using OpenCV.
    Plays until the end or until 'q' is pressed.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return

    print(f"Playing video: {video_path}")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Video", frame)
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


def delete_video(video_path):
    """
    Deletes a local video file (e.g., one downloaded by load_video()).
    Only allows deletion inside the temp_videos folder for safety.
    """
    abs_path = os.path.abspath(video_path)
    temp_dir_abs = os.path.abspath(TEMP_DIR)

    # Ensure safety: only delete inside temp_videos
    if abs_path.startswith(temp_dir_abs) and os.path.exists(abs_path):
        os.remove(abs_path)
        print(f"Deleted: {abs_path}")
    else:
        print(f"Skipped deletion (unsafe path): {abs_path}")


# === Example usage ===
# import pandas as pd
# df = pd.read_hdf("nexar_train_prepared.h5", key="data")
# row = df.iloc[0]
# local_path = load_video(row)
# script1(local_path)
# delete_video(local_path)


### Example Usage

In [ ]:
import pandas as pd
df = pd.read_hdf("nexar_train_prepared.h5", key="data")
df

In [ ]:
row = df.iloc[0]
local_path = load_video(row)
script1(local_path)
delete_video(local_path)

In [ ]:
import pandas as pd
import concurrent.futures

def script2(video_path):
    """
    Return the length of a video in seconds as a float.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps <= 0 or frame_count <= 0:
        return None
    return float(frame_count / fps)


# === Batch processing ===
input_path = "nexar_train_prepared.h5"
output_path = "output.h5"
df = pd.read_hdf(input_path, key="data")
results = []
n = 10  # number of clips per batch
for i in range(3):  # run 3 batches
    start = i * n
    end = start + n
    if start >= len(df):
        break
    batch = df.iloc[start:end]
    # Load videos concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as executor:
        futures = {executor.submit(load_video, row): idx for idx, row in batch.iterrows()}
        local_paths = {}
        for future in concurrent.futures.as_completed(futures):
            idx = futures[future]
            try:
                local_paths[idx] = future.result()
            except Exception as e:
                print(f"Error loading row {idx}: {e}")
    for idx, local_path in local_paths.items():
        clip_length = script2(local_path)
        row_data = batch.loc[idx].to_dict()
        row_data["output"] = clip_length
        results.append(row_data)
        delete_video(local_path)

output_df = pd.DataFrame(results)
output_df.to_hdf(output_path, key="data", mode="w")

print(f"✅ Output saved to {output_path}")
print(output_df.head())



Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00408.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00128.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00072.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00471.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00205.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00822.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00208.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00457.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00900.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00948.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00641.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00841.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00938.mp4
Deleted: /Users/aneeshkalla/Desktop/A